# Експеримент 1: інші готові моделі та використання аліасів навичок

**Питання:** наскільки можна покращити zero-shot cosine baseline (без навчання), якщо
1. взяти іншу готову embedding-модель (`all-mpnet-base-v2`, доменна `TechWolf/JobBERT-v2`);
2. використати **усі аліаси навички**, а не лише перший.

**Стратегії для аліасів навички** (у dev-запитах для посад є лише одна назва, тож аліаси є тільки в навичок):
- `first` — лише `skill_aliases[0]` (як у baseline);
- `mean` — усереднений embedding усіх аліасів;
- `max` — скор навички = максимум cosine по її аліасах.

Метрики — ті самі, що в baseline (`ranking_metrics.py`), оцінка на dev по всьому корпусу.

In [1]:
import os, sys
from pathlib import Path

# ноутбук лежить в experiments/, а config.py та ranking_metrics.py — в корені проєкту
if not Path("config.py").exists():
    os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Робоча папка:", os.getcwd())

Робоча папка: /Users/mac/spj/project/Job-Skill-Matching-Ranking-and-Classification-Using-NLP-Approaches


In [2]:
import ast
import json
import gc

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from ranking_metrics import evaluate_query, save_results

VAL_DIR = Path("data/raw/TaskB/validation")

# коротка назва (для імені файлу) -> назва моделі на Hugging Face
MODELS = {
    "minilm": "all-MiniLM-L6-v2",
    "mpnet": "all-mpnet-base-v2",
    "jobbert": "TechWolf/JobBERT-v2",
}
ALIAS_MODES = ["first", "mean", "max"]

## 1. Дані (ті самі, що в baseline)

In [3]:
queries = pd.read_csv(VAL_DIR / "queries", sep="\t").rename(columns={"jobtitle": "job_name", "q_id": "job_id"})
corpus = pd.read_csv(VAL_DIR / "corpus_elements", sep="\t").rename(columns={"c_id": "skill_id"})
corpus["skill_aliases"] = corpus["skill_aliases"].apply(ast.literal_eval)
qrels = pd.read_csv(VAL_DIR / "qrels.tsv", header=None, sep="\t", names=["job_id", "_", "skill_id", "label"])

skill_index = {sid: i for i, sid in enumerate(corpus["skill_id"])}
grades_by_job = {}
for r in qrels.itertuples(index=False):
    grades_by_job.setdefault(r.job_id, np.zeros(len(corpus)))[skill_index[r.skill_id]] = r.label

# усі аліаси підряд; starts[i] — позиція першого аліасу i-ої навички
alias_counts = corpus["skill_aliases"].apply(len).values
assert alias_counts.min() >= 1
alias_texts = [a for aliases in corpus["skill_aliases"] for a in aliases]
starts = np.r_[0, np.cumsum(alias_counts)[:-1]]
owner = np.repeat(np.arange(len(corpus)), alias_counts)  # номер навички для кожного аліасу

print(f"queries: {len(queries)}, skills: {len(corpus)}, aliases: {len(alias_texts)} "
      f"(в середньому {alias_counts.mean():.1f} на навичку)")

queries: 304, skills: 9052, aliases: 64321 (в середньому 7.1 на навичку)


## 2. Експеримент
Для кожної моделі: кодуємо посади та всі аліаси (унікальні тексти), потім оцінюємо три стратегії.
Якщо якусь модель не вдалось завантажити, ноутбук виведе помилку й піде до наступної.

In [4]:
def encode_texts(model, texts):
    unique = sorted(set(texts))
    vecs = model.encode(unique, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    return dict(zip(unique, vecs))


def evaluate(score_fn, job_emb):
    per_query = []
    for row in queries.itertuples(index=False):
        if row.job_id not in grades_by_job:
            continue
        per_query.append(evaluate_query(score_fn(job_emb[row.job_name]), grades_by_job[row.job_id]))
    return per_query


for short, model_name in MODELS.items():
    print(f"\n===== {model_name} =====")
    try:
        model = SentenceTransformer(model_name)
        job_emb = encode_texts(model, queries["job_name"].tolist())
        alias_map = encode_texts(model, alias_texts)
    except Exception as e:  # модель недоступна або має інший інтерфейс
        print(f"ПРОПУСКАЮ {model_name}: {type(e).__name__}: {e}")
        continue

    A = np.vstack([alias_map[t] for t in alias_texts])  # (кількість аліасів, dim)

    # mean: усереднюємо вектори аліасів навички й нормалізуємо назад до довжини 1
    A_mean = np.zeros((len(corpus), A.shape[1]), dtype=np.float32)
    np.add.at(A_mean, owner, A)
    A_mean /= np.linalg.norm(A_mean, axis=1, keepdims=True)

    scorers = {
        "first": lambda e: A[starts] @ e,
        "mean": lambda e: A_mean @ e,
        "max": lambda e: np.maximum.reduceat(A @ e, starts),
    }
    descriptions = {
        "first": f"Zero-shot cosine ({model_name}); навичка представлена першим аліасом.",
        "mean": f"Zero-shot cosine ({model_name}); навичка = усереднений embedding усіх аліасів.",
        "max": f"Zero-shot cosine ({model_name}); скор навички = максимум cosine по всіх її аліасах.",
    }
    for mode in ALIAS_MODES:
        per_query = evaluate(scorers[mode], job_emb)
        save_results(f"cosine_{short}_{mode}", descriptions[mode], per_query)

    del model, A, A_mean, alias_map
    gc.collect()


===== all-MiniLM-L6-v2 =====


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/986 [00:00<?, ?it/s]

Збережено: results/cosine_minilm_first.json
Збережено: results/cosine_minilm_mean.json
Збережено: results/cosine_minilm_max.json

===== all-mpnet-base-v2 =====


...

2_Asym/140216480445248_Dense/model.safet(…): reconstructing file:   0%|          |  0.00B / 3.15MB            

2_Asym/140216480445248_Dense/model.safet(…): downloading bytes:           |  0.00B            

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/986 [00:00<?, ?it/s]

Збережено: results/cosine_jobbert_first.json
Збережено: results/cosine_jobbert_mean.json
Збережено: results/cosine_jobbert_max.json


## 3. Порівняння
Перевірка коректності: `cosine_minilm_first` має збігатися з `cosine_minilm` з baseline-ноутбука (nDCG graded ≈ 0.615).

In [5]:
rows = []
for p in sorted(Path("results").glob("*.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    rows.append({"approach": p.stem, **{k: v for k, v in d.items() if k != "description"}})

pd.DataFrame(rows).sort_values("ndcg_graded", ascending=False).reset_index(drop=True)

,approach,ndcg_graded,ndcg_binary,map_binary,mrr_binary,p@1,p@5,p@10
0,cosine_jobbert_mean,0.7381,0.7889,0.3416,0.7886,0.6480,0.5632,0.5178
1,cosine_jobbert_first,0.7359,0.7899,0.3479,0.8210,0.6974,0.5645,0.4914
2,cosine_jobbert_max,0.7223,0.7723,0.3077,0.7535,0.5987,0.5125,0.4816
3,cosine_mpnet_mean,0.6327,0.6763,0.1612,0.6302,0.4539,0.3980,0.3711
4,cosine_mpnet_first,0.6251,0.6704,0.1545,0.6135,0.4408,0.3809,0.3549
5,cosine_minilm_mean,0.6214,0.6645,0.1477,0.6397,0.4803,0.3836,0.3658
6,cosine_mpnet_max,0.6189,0.6625,0.1446,0.6059,0.4178,0.3783,0.3523
7,cosine_minilm,0.6154,0.6599,0.1430,0.6206,0.4342,0.3829,0.3539
8,cosine_minilm_first,0.6154,0.6599,0.1430,0.6206,0.4342,0.3829,0.3539
9,cosine_minilm_max,0.6105,0.6536,0.1360,0.5901,0.3882,0.3730,0.3464
